# Customer Churn Prediction Using Artificial Neural Networks

A TensorFlow/Keras binary-classification project for predicting whether a bank customer is likely to leave.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# 1. Import Libraries
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
import seaborn as sns

# 2. Load Dataset
data = pd.read_csv("Churn_Modelling.csv")
print(data.head())

# 3. Feature Matrix X and Target Y
X = data.iloc[:, 3:-1].values   # columns 4 to 13 (independent)
Y = data.iloc[:, -1].values     # last column (Exited)

# 4. Encode Gender (Label Encoding)
LE1 = LabelEncoder()
X[:, 2] = np.array(LE1.fit_transform(X[:, 2]))

# 5. Encode Geography (One-Hot Encoding)
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [1])], remainder="passthrough")
X = np.array(ct.fit_transform(X))

# 6. Train-Test Split (80/20)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

# 7. Feature Scaling
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# 8. Build ANN
ann = tf.keras.models.Sequential()
ann.add(tf.keras.layers.Dense(units=6, activation="relu"))  # Hidden Layer 1
ann.add(tf.keras.layers.Dense(units=6, activation="relu"))  # Hidden Layer 2
ann.add(tf.keras.layers.Dense(units=1, activation="sigmoid"))  # Output Layer

# 9. Compile ANN
ann.compile(optimizer="adam", loss="binary_crossentropy", metrics=['accuracy'])

# 10. Train ANN
history = ann.fit(X_train, Y_train, batch_size=32, epochs=100)

# 11. Plot Training Accuracy
plt.plot(history.history['accuracy'])
plt.title('Model Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.grid(True)
plt.savefig('accuracy_plot.png')
plt.show()

# 12. Plot Training Loss
plt.plot(history.history['loss'], color='red')
plt.title('Model Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.savefig('loss_plot.png')
plt.show()

# 13. Predictions & Confusion Matrix
Y_pred = ann.predict(X_test)
Y_pred = (Y_pred > 0.5)
cm = confusion_matrix(Y_test, Y_pred)
print("Confusion Matrix:\n", cm)
print("Accuracy:", accuracy_score(Y_test, Y_pred))

# Plot Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig('confusion_matrix.png')
plt.show()

# 14. Single Prediction Example
sample = sc.transform([[1, 0, 0, 600, 1, 40, 3, 60000, 2, 1, 1, 50000]])
result = ann.predict(sample) > 0.5
print("Will customer exit?", result)